# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahsan-Qadeer/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

## 1. My rule and its reason codes

Pages that have low clicks but large number of appearances (impressions) would be great for review since making them better could improve traffic. I have also used page views as a signal so that low traffic pages are not prioritized over higher impact pages.

###Reasons codes:
LOW_CTR_HIGH_IMPRESSIONS

###Action:
REVIEW_CONTENT



In [14]:
pageviews = con.sql(f"""
SELECT
CASE
    WHEN ga4_pageviews < 100 THEN '<100'
    WHEN ga4_pageviews < 1000 THEN '100-999'
    WHEN ga4_pageviews < 10000 THEN '1000-9999'
    ELSE '10000+'
END AS pageview_bucket,
COUNT(*) AS n
FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
GROUP BY pageview_bucket
ORDER BY pageview_bucket;
""").df()

pageviews

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pageview_bucket,n
0,100-999,324
1,<100,413642


In [15]:
impressions = con.sql(f"""
SELECT
CASE
WHEN gsc_impressions < 100 THEN '<100'
WHEN gsc_impressions < 1000 THEN '100-999'
WHEN gsc_impressions < 10000 THEN '1000-9999'
ELSE '10000+'
END AS impression_bucket,
COUNT(*) AS n
FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
GROUP BY impression_bucket
ORDER BY impression_bucket
""").df()

print(impressions)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  impression_bucket        n
0           100-999   606189
1         1000-9999    32291
2            10000+      128
3              <100  2972453


The impression counts and page views are in several buckets. This implies that the signals have enough variation to allow a basis for page review prioritization

## 2. Build the ranked queue (writes the CSV)

In the ranked queue, pages are ranked based on a simple score using search visbility and ctr. Higher impression with lower CTR pages are nearer to the queue start while pages with lesser clicks are prioritised over pages that already have many clicks. Lower than 5% CTR receive reason code HIGH_IMPRESSIONS_LOW_CTR and action label REEVIEW_CONTENT. All the other pages get action label MONITOR.

In [16]:
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)

con.sql(f"""
COPY (

SELECT

report_date,
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,

CASE
WHEN gsc_impressions>0
THEN CAST(gsc_clicks AS DOUBLE)/gsc_impressions
ELSE 0
END AS ctr,

gsc_impressions *
(
1-
CASE
WHEN gsc_impressions>0
THEN CAST(gsc_clicks AS DOUBLE)/gsc_impressions
ELSE 0
END
) AS action_score,

'LOW_CTR_HIGH_IMPRESSIONS' AS reason_code,

'Review content for refresh' AS action

FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')

WHERE gsc_data_available IS TRUE

ORDER BY action_score DESC

)

TO 'work/outputs/baseline_action_score.csv'

(HEADER, DELIMITER ',')
""")

print("CSV written successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV written successfully.


## 3. Top-20 review

| Rank | Action | Reason code | Confidence | What would make it wrong |
|------|--------|-------------|------------|--------------------------|
| 1 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | CTR may be low because the search query is not relevant rather than the page quality. |
| 2 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | The page may already rank well for informational searches where naturally few users click. |
| 3 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Low CTR could be caused by misleading search snippets instead of outdated content. |
| 4 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Same page appears on another day; repeated appearance does not necessarily indicate a new issue. |
| 5 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Zero clicks may be due to temporary search behavior rather than poor content. |
| 6 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | High impressions alone do not guarantee that refreshing the page will improve traffic. |
| 7 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Seasonal search demand may temporarily reduce CTR. |
| 8 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | The page could target highly competitive keywords where low CTR is expected. |
| 9 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | External factors such as SERP features may reduce clicks despite good content. |
| 10 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | The page may simply require a better title or meta description rather than a full refresh. |
| 11 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Duplicate appearance across multiple dates may overstate its importance. |
| 12 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | The page may already satisfy user intent despite low CTR. |
| 13 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | CTR may improve naturally as rankings change without requiring intervention. |
| 14 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | The page may have unusually high impressions from a short-lived trend. |
| 15 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Search demand fluctuations could explain the observed performance. |
| 16 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Zero clicks may result from a very broad search query rather than poor page quality. |
| 17 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Refreshing the content may not improve CTR if ranking position is unchanged. |
| 18 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | The page may require technical SEO improvements instead of content changes. |
| 19 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | Small day-to-day variations should not be interpreted as a long-term trend. |
| 20 | Review content for refresh | LOW_CTR_HIGH_IMPRESSIONS | High | This rule uses only impressions and CTR, so pages with low visibility but high potential may be missed. || Rank | Action | Reason code | Confidence | What would make it wrong |


In [17]:
top20 = con.sql("""
SELECT *
FROM read_csv_auto('work/outputs/baseline_action_score.csv')
ORDER BY action_score DESC
LIMIT 20
""").df()

top20

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,action_score,reason_code,action
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.000025,40083.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,0.006411,39053.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
2,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,0.000051,39001.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
3,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,0.007051,38165.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
4,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,0.000000,37368.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
5,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,0.006355,35179.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,0.006405,34594.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
7,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,0.006791,34371.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
8,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.000000,33383.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh
9,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,215,0.006404,33356.0,LOW_CTR_HIGH_IMPRESSIONS,Review content for refresh


## 4. Weak picks + leakage check

The weakest picks are pages with extremely high impressions but zero or almost zero CTR. Although these pages are reasonable candidates for review, low CTR alone does not prove that the content is poor. Other factors such as search intent, SERP features, or highly competitive queries may explain the low click-through rate.

The baseline rule also ranks the same content on multiple report dates because the scoring is performed on daily observations. In a production system, these repeated daily entries could be consolidated into a single review recommendation.

### Leakage check

- No trend labels were used.
- No future performance metrics or outcome variables were used.
- The rule only uses observable Google Search Console signals (impressions and clicks) available at the decision time.
- Therefore, no label-derived information leaked into the baseline score.

In [18]:
weak = top20.nsmallest(5, "ctr")[[
    "content_hash_id",
    "gsc_impressions",
    "ctr",
    "reason_code"
]]

weak

,content_hash_id,gsc_impressions,ctr,reason_code
4,content_945d6ff91386c817,37368,0.000000,LOW_CTR_HIGH_IMPRESSIONS
8,content_fec55986a1868d62,33383,0.000000,LOW_CTR_HIGH_IMPRESSIONS
10,content_44f34c0a90047651,32958,0.000000,LOW_CTR_HIGH_IMPRESSIONS
15,content_fec55986a1868d62,31472,0.000000,LOW_CTR_HIGH_IMPRESSIONS
0,content_44f34c0a90047651,40084,0.000025,LOW_CTR_HIGH_IMPRESSIONS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.